# DocClean-Net — Colab Training Notebook

Entrena el U-Net de DocClean-Net en una GPU T4 gratuita de Google Colab.

**Flujo completo:**
1. Verificar GPU
2. Clonar repo desde GitHub
3. Instalar dependencias
4. Generar 10 000 pares sintéticos
5. Inspección visual del dataset
6. Entrenar 50 épocas
7. Descargar `best.pt` y `train_log.csv`

**Tiempo estimado:** 20-35 minutos (GPU T4).


## 0 · Verificar GPU

In [ ]:
import torch

gpu_info = !nvidia-smi
print("\n".join(gpu_info))
print()

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU disponible: {name} ({mem:.1f} GB)")
else:
    print("✗ No hay GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno → GPU T4")
    raise RuntimeError("GPU requerida para este notebook.")


## 1 · Clonar repositorio

In [ ]:
from pathlib import Path

DIRTY_DIR = Path("data/synthetic/dirty")
N_PAIRS   = 10_000

n_existing = len(list(DIRTY_DIR.glob("*.png"))) if DIRTY_DIR.exists() else 0
print(f"Pares existentes: {n_existing}")

if n_existing >= N_PAIRS:
    print(f"✓ Dataset completo ({n_existing} pares). Saltando generación.")
else:
    print(f"Generando {N_PAIRS} pares desde cero (sobreescribe si ya existen)...")
    !python -m data.generate_dataset \
        --n       {N_PAIRS} \
        --output  data/synthetic/ \
        --size    512 \
        --seed    42 \
        --workers 4

    final = len(list(DIRTY_DIR.glob("*.png")))
    print(f"\n✓ Dataset listo: {final} pares en data/synthetic/")


## 2 · Instalar dependencias

In [ ]:
# torch ya viene preinstalado en Colab con CUDA.
# Solo instalamos lo que falta.
!pip install pytorch-msssim ttkbootstrap --quiet

# Verificar imports críticos
import cv2, numpy as np
from model.unet import UNet
from model.losses import CombinedLoss
from model.dataset import DocCleanDataset

net = UNet()
n_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"✓ UNet cargado — {n_params:,} parámetros entrenables")
print(f"✓ OpenCV {cv2.__version__}")
print(f"✓ NumPy {np.__version__}")
print(f"✓ PyTorch {__import__('torch').__version__}")


## 3 · Generar dataset sintético (10 000 pares)

Se generan directamente en Colab — más rápido que subir desde local.
Los pares se guardan en `data/synthetic/dirty/` y `data/synthetic/clean/`.

**Tiempo estimado:** 5-10 minutos en CPU (la generación no usa GPU).


In [ ]:
import os
from pathlib import Path

DIRTY_DIR = Path("data/synthetic/dirty")
CLEAN_DIR = Path("data/synthetic/clean")
N_PAIRS   = 10_000

n_existing = len(list(DIRTY_DIR.glob("*.png"))) if DIRTY_DIR.exists() else 0
print(f"Pares ya existentes: {n_existing}")

if n_existing >= N_PAIRS:
    print(f"✓ Dataset completo ({n_existing} pares). Saltando generación.")
else:
    to_generate = N_PAIRS - n_existing
    print(f"Generando {to_generate} pares nuevos...")
    !python -m data.generate_dataset --n {to_generate} --output data/synthetic/ --offset {n_existing}
    final = len(list(DIRTY_DIR.glob("*.png")))
    print(f"✓ Dataset listo: {final} pares en data/synthetic/")


## 4 · Inspección visual del dataset

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
from pathlib import Path

dirty_dir = Path("data/synthetic/dirty")
clean_dir = Path("data/synthetic/clean")

# Mostrar 4 pares aleatorios
rng = np.random.default_rng(seed=0)
all_dirty = sorted(dirty_dir.glob("*.png"))
indices   = rng.choice(len(all_dirty), size=4, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Dataset sintético — 4 pares aleatorios (izq: dirty, der: clean)",
             fontsize=13)

for col, idx in enumerate(indices):
    dirty_path = all_dirty[idx]
    clean_path = clean_dir / dirty_path.name.replace("dirty_", "clean_")

    dirty_bgr = cv2.imread(str(dirty_path))
    clean_bgr = cv2.imread(str(clean_path))

    axes[0, col].imshow(cv2.cvtColor(dirty_bgr, cv2.COLOR_BGR2RGB))
    axes[0, col].set_title(f"dirty {idx}", fontsize=9)
    axes[0, col].axis("off")

    axes[1, col].imshow(cv2.cvtColor(clean_bgr, cv2.COLOR_BGR2RGB))
    axes[1, col].set_title(f"clean {idx}", fontsize=9)
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()


## 5 · Entrenar 50 épocas

- Split: 90% train (9000) / 10% val (1000), semilla fija 42
- Optimizador: AdamW lr=1e-3, weight_decay=1e-4
- Scheduler: CosineAnnealingLR → lr decae a 1e-5 en época 50
- Loss: CombinedLoss = MSE×0.7 + (1−SSIM)×0.3
- Checkpoint: `checkpoints/best.pt` (mejor val_loss)

**Tiempo estimado en T4: 20-30 minutos.**


In [ ]:
!python -m model.train \
    --data      data/synthetic/ \
    --epochs    50 \
    --batch     16 \
    --lr        1e-3 \
    --patch-size 256 \
    --checkpoint-dir checkpoints/


## 6 · Curvas de entrenamiento

In [ ]:
import csv
import matplotlib.pyplot as plt
from pathlib import Path

log_path = Path("checkpoints/train_log.csv")
if not log_path.exists():
    print("No se encontró train_log.csv — ejecuta el entrenamiento primero.")
else:
    rows = list(csv.DictReader(log_path.read_text().splitlines()))
    epochs     = [int(r["epoch"])      for r in rows]
    train_loss = [float(r["train_loss"]) for r in rows]
    val_loss   = [float(r["val_loss"])   for r in rows]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epochs, train_loss, label="train_loss", linewidth=2)
    ax.plot(epochs, val_loss,   label="val_loss",   linewidth=2, linestyle="--")
    ax.set_xlabel("Época")
    ax.set_ylabel("CombinedLoss")
    ax.set_title("DocClean-Net — Curvas de entrenamiento")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    best_epoch = epochs[val_loss.index(min(val_loss))]
    print(f"Mejor val_loss: {min(val_loss):.6f} en época {best_epoch}")
    print(f"Train loss final: {train_loss[-1]:.6f}")


## 7 · Descargar checkpoint y log

Descarga `best.pt` y `train_log.csv` a tu PC.
Coloca `best.pt` en `checkpoints/` de tu repo local.


In [ ]:
from google.colab import files
import os

checkpoint = "checkpoints/best.pt"
log        = "checkpoints/train_log.csv"

if os.path.exists(checkpoint):
    files.download(checkpoint)
    print(f"✓ Descargando {checkpoint}")
else:
    print("✗ No se encontró best.pt — verifica que el entrenamiento terminó correctamente.")

if os.path.exists(log):
    files.download(log)
    print(f"✓ Descargando {log}")


## 8 · Test rápido de inferencia (opcional)

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from model.unet import UNet

# Carga el modelo entrenado
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = UNet().to(device)
model.load_state_dict(torch.load("checkpoints/best.pt", map_location=device))
model.eval()

# Toma un par dirty/clean aleatorio como muestra
dirty_dir = Path("data/synthetic/dirty")
sample    = sorted(dirty_dir.glob("*.png"))[42]
clean_path = Path("data/synthetic/clean") / sample.name.replace("dirty_", "clean_")

dirty_bgr = cv2.imread(str(sample))
clean_bgr = cv2.imread(str(clean_path))

# Preprocesado idéntico al DocCleanDataset
gray  = cv2.cvtColor(dirty_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
x     = torch.from_numpy(gray).unsqueeze(0).unsqueeze(0).to(device)  # (1,1,H,W)

# Toma un patch central 256×256
h, w  = gray.shape
top   = (h - 256) // 2
left  = (w - 256) // 2
x     = x[:, :, top:top+256, left:left+256]

with torch.no_grad():
    pred = model(x)

pred_np = pred.squeeze().cpu().numpy()  # (256, 256) float32

# Visualización comparativa
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(dirty_bgr[top:top+256, left:left+256], cv2.COLOR_BGR2RGB))
axes[0].set_title("Dirty (entrada)")
axes[0].axis("off")

axes[1].imshow(pred_np, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("U-Net output")
axes[1].axis("off")

axes[2].imshow(cv2.cvtColor(clean_bgr[top:top+256, left:left+256], cv2.COLOR_BGR2RGB))
axes[2].set_title("Clean (ground truth)")
axes[2].axis("off")

plt.tight_layout()
plt.show()
print(f"Output — min: {pred_np.min():.4f}, max: {pred_np.max():.4f}, mean: {pred_np.mean():.4f}")
